# LiveMatch Edge - Analysis Workflow

This notebook preprocesses in-play football data, trains outcome classifiers, and evaluates confidence-based live betting rules. The raw course dataset is intentionally not committed to the public repository; place it under `data/match_data_582_converted.csv`, keep it beside this notebook, or set `MATCH_DATA_CSV` before running.

In [ ]:

from pathlib import Path
import os
import pandas as pd

CANDIDATE_DATA_PATHS = [
    Path(os.environ.get("MATCH_DATA_CSV", "")) if os.environ.get("MATCH_DATA_CSV") else None,
    Path("../data/match_data_582_converted.csv"),
    Path("data/match_data_582_converted.csv"),
    Path("match_data_582_converted.csv"),
]

data_path = next((path for path in CANDIDATE_DATA_PATHS if path and path.exists()), None)
if data_path is None:
    searched = "\n".join(f"- {path}" for path in CANDIDATE_DATA_PATHS if path)
    raise FileNotFoundError(
        "LiveMatch Edge requires the original course CSV, which is not committed to this public repository.\n"
        "Place the file at ../data/match_data_582_converted.csv, data/match_data_582_converted.csv, "
        "or set MATCH_DATA_CSV to its absolute path.\n\n"
        f"Searched:\n{searched}"
    )

match_df = pd.read_csv(data_path)
print(f"Loaded {data_path} with {len(match_df):,} rows and {match_df.shape[1]} columns.")
display(match_df.head())


In [ ]:

# To ensure time order
match_data_sorted = match_df.sort_values(['fixture_id', 'halftime', 'minute', 'second'])
match_data_sorted.head()

In [ ]:
# Forward-fill live features within each fixture only, then fill remaining startup missing values with 0.
# Grouped fill avoids leaking values from one match into another match.
columns_to_fill = match_data_sorted.columns[15:]
match_data_sorted[columns_to_fill] = (
    match_data_sorted
    .groupby("fixture_id", group_keys=False)[columns_to_fill]
    .ffill()
    .fillna(0)
)

match_data_sorted.head()


In [ ]:
# Adjust minutes where halftime is "2nd-half"
match_data_sorted.loc[match_data_sorted['halftime'] == '2nd-half', 'minute'] += 45



In [ ]:
match_data_sorted['P(1)_v1'] = 1 / match_data_sorted['1']
match_data_sorted['P(X)_v1'] = 1 / match_data_sorted['X']
match_data_sorted['P(2)_v1'] = 1 / match_data_sorted['2']

# Calculate the sum of reciprocal odds
match_data_sorted['match_total_prob'] = match_data_sorted['P(1)_v1'] + match_data_sorted['P(X)_v1'] + match_data_sorted['P(2)_v1']

# Normalize each probability
match_data_sorted['1'] = match_data_sorted['P(1)_v1'] / match_data_sorted['match_total_prob']
match_data_sorted['X'] = match_data_sorted['P(X)_v1'] / match_data_sorted['match_total_prob']
match_data_sorted['2'] = match_data_sorted['P(2)_v1'] / match_data_sorted['match_total_prob']

# Drop intermediate columns if not needed
match_data_sorted.drop(columns=['P(1)_v1', 'P(X)_v1', 'P(2)_v1', 'match_total_prob'], inplace=True)

# Display the first few rows to verify the optimization
match_data_sorted[['fixture_id', 'minute', '1', 'X', '2']].head()


In [ ]:
# List of percentage columns to normalize
percentage_columns = [
    'Ball Possession % - away',
    'Ball Possession % - home',
    'Successful Passes Percentage - away',
    'Successful Passes Percentage - home'
]

# Normalize percentage columns to a range [0, 1] by dividing by 100
match_data_sorted[percentage_columns] = match_data_sorted[percentage_columns] / 100

# Display the first few rows to verify normalization
match_data_sorted[['fixture_id', 'minute'] + percentage_columns].head()



In [ ]:
# Function to list continuous features
def list_continuous_features(df):
    # Exclude non-continuous columns such as fixture_id, minute, odds, and result
    excluded_columns = ['fixture_id', 'minute', '1', 'X', '2', 'result']
    continuous_features = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] and col not in excluded_columns]
    return continuous_features

# Get the list of continuous features
continuous_features = list_continuous_features(match_data_sorted)

# Display the list of continuous features
continuous_features



In [ ]:
# Normalize paired home/away continuous features using the original pair denominator.
# This avoids changing the away denominator after the home column has already been overwritten.
normalized_data = match_data_sorted.copy()
for feature in continuous_features:
    if "home" in feature and feature.replace("home", "away") in continuous_features:
        away_feature = feature.replace("home", "away")
        home_values = match_data_sorted[feature].astype(float)
        away_values = match_data_sorted[away_feature].astype(float)
        denominator = home_values + away_values
        denominator = denominator.replace(0, 1e-10)
        normalized_data[feature] = home_values / denominator
        normalized_data[away_feature] = away_values / denominator

normalized_data = normalized_data[["fixture_id", "minute"] + continuous_features + ["1", "X", "2", "result"]].copy()
normalized_data.head()


In [ ]:
# Add 'match_start_datetime' to the DataFrame for splitting
normalized_data['match_start_datetime'] = match_data_sorted['match_start_datetime']

# Convert 'match_start_datetime' to datetime for easier splitting
normalized_data['match_start_datetime'] = pd.to_datetime(normalized_data['match_start_datetime'])

# Split the data based on the date "2024-11-01"
train_data = normalized_data[normalized_data['match_start_datetime'] < '2024-11-01'].copy()
test_data = normalized_data[normalized_data['match_start_datetime'] >= '2024-11-01'].copy()

# Drop 'match_start_datetime' from both sets after splitting
train_data.drop(columns=['match_start_datetime'], inplace=True)
test_data.drop(columns=['match_start_datetime'], inplace=True)

unique_fixtures_test = test_data['fixture_id'].nunique()
unique_fixtures_test

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder



In [ ]:
label_encoder = LabelEncoder()
train_data['encoded_result'] = label_encoder.fit_transform(train_data['result'])
test_data['encoded_result'] = label_encoder.transform(test_data['result'])

# Prepare X_train, y_train, X_test, y_test
X_train = train_data.drop(columns=['result', 'fixture_id', 'encoded_result'])
y_train = train_data['encoded_result']
X_test = test_data.drop(columns=['result', 'fixture_id', 'encoded_result'])
y_test = test_data['encoded_result']



In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
model = XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=4)  

# Updated parameter grid for XGBoost
param_grid_xgb = {
    'n_estimators': [100, 150],  # Number of trees
    'learning_rate': [0.05, 0.1],  # Shrinkage step size
    'max_depth': [3, 5],  # Tree depth to control overfitting
    'min_child_weight': [1, 5],  # Minimum sum of instance weights for a child node
    'gamma': [0, 0.1],  # Minimum loss reduction required to make a further split
    'subsample': [0.8, 1.0],  # Fraction of samples used for training each tree
    'colsample_bytree': [0.8, 1.0],  # Fraction of features used for each tree
}

# GridSearchCV with XGBoost
grid_search_xgb = GridSearchCV(
    estimator=model,
    param_grid=param_grid_xgb,
    cv=3,  # 3-fold cross-validation
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

# Check if the sample size doesn't exceed available data
sample_size = min(20000, len(X_train))
X_train_sampled = X_train.sample(n=sample_size, random_state=42)
y_train_sampled = y_train.loc[X_train_sampled.index]

# Perform the Grid Search
grid_search_xgb.fit(X_train_sampled, y_train_sampled)

# Get the best model, parameters, and score
best_model_xgb = grid_search_xgb.best_estimator_
best_params_xgb = grid_search_xgb.best_params_
best_score_xgb = grid_search_xgb.best_score_

print("Best Params:", best_params_xgb)
print("Best Score:", best_score_xgb)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = best_model_xgb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {acc:.2f}")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.show()


In [ ]:
from sklearn.linear_model import LogisticRegression

#  Logistic Regression (GLM) model with regularization
model_glm = LogisticRegression(multi_class='multinomial', solver='saga', max_iter=1000)

# Hyperparameter grid for regularization strength
param_grid_glm = {
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength (inverse of lambda)
    'penalty': ['l1', 'l2'],  # L1 for Lasso, L2 for Ridge regularization
}

# Grid search for GLM
grid_search_glm = GridSearchCV(estimator=model_glm, param_grid=param_grid_glm, cv=3, n_jobs=-1, verbose=1)
grid_search_glm.fit(X_train, y_train)

# Get best model, parameters, and score
best_model_glm = grid_search_glm.best_estimator_
best_params_glm = grid_search_glm.best_params_
best_score_glm = grid_search_glm.best_score_

print("Best Params for GLM:", best_params_glm)
print("Best Score for GLM:", best_score_glm)



In [ ]:

# Predict on test set
y_pred_glm = best_model_glm.predict(X_test)

# Evaluate GLM performance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

accuracy_glm = accuracy_score(y_test, y_pred_glm)
print(f"GLM Test Set Accuracy: {accuracy_glm:.2f}")
print(classification_report(y_test, y_pred_glm))

# Confusion Matrix for GLM
cm_glm = confusion_matrix(y_test, y_pred_glm)
sns.heatmap(cm_glm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix for GLM')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier



# Random Forest Classifier
model_rf = RandomForestClassifier(random_state=42)

# Updated hyperparameter grid
param_grid_rf = {
    'n_estimators': [100, 150],  # Number of trees
    'max_depth': [10, 15, 20],  # Focus on key depths
    'min_samples_split': [2, 5],  # Minimum samples required for split
    'min_samples_leaf': [1, 2],  # Minimum samples per leaf
    'bootstrap': [True]  # Only test bootstrap=True to reduce combinations
}

# GridSearchCV for Random Forest
grid_search_rf = GridSearchCV(
    estimator=model_rf,
    param_grid=param_grid_rf,
    cv=3,  # 3-fold cross-validation
    n_jobs=4,  # Use 4 cores to balance load
    verbose=1
)

# Fit with reduced grid
grid_search_rf.fit(X_train, y_train)

# Get best model, parameters, and score
best_model_rf = grid_search_rf.best_estimator_
best_params_rf = grid_search_rf.best_params_
best_score_rf = grid_search_rf.best_score_

print("Best Params for Random Forest:", best_params_rf)
print("Best Score for Random Forest:", best_score_rf)




In [ ]:
best_model_rf.fit(X_train, y_train)

# Predict on the test set
y_pred_rf = best_model_rf.predict(X_test)

# Calculate accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Test Set Accuracy: {accuracy_rf:.2f}")

# Print classification report
print(classification_report(y_test, y_pred_rf))

# Plot confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix for Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Revised live betting strategy
import pandas as pd
def revised_betting_strategy(model, test_df, initial_confidence_threshold=0.7, cutoff_minute=90, confidence_step=0.01):
    decisions = {}

    for fixture_id in test_df['fixture_id'].unique():
        # Filter data for the specific fixture and sort by time
        fixture_data = test_df[test_df['fixture_id'] == fixture_id].sort_values(by='minute')
        current_confidence_threshold = initial_confidence_threshold

        for _, row in fixture_data.iterrows():
            if row['minute'] > cutoff_minute:
                break  # Do not bet after cutoff minute

            # Extract features up to the current minute
            X_single = row.drop(['result', 'fixture_id', 'encoded_result']).values.reshape(1, -1)
            pred_proba = model.predict_proba(X_single)[0]  # Get probability predictions
            pred_class = model.classes_[pred_proba.argmax()]  # Get predicted class (1, 2, or X)
            max_proba = pred_proba.max()  # Maximum probability (confidence)

            # Adjust confidence threshold slightly as the game progresses
            current_confidence_threshold += confidence_step * (row['minute'] / 90)

            # If confidence exceeds the adjusted threshold, make a prediction
            if max_proba >= current_confidence_threshold:
                decisions[fixture_id] = {
                    'minute': row['minute'],
                    'prediction': label_encoder.inverse_transform([pred_class])[0],
                    'confidence': max_proba
                }
                break  # Stop after making the first confident decision

        # If no confident prediction was made during the entire match
        if fixture_id not in decisions:
            decisions[fixture_id] = {
                'minute': None,
                'prediction': "no action",
                'confidence': None
            }

    return pd.DataFrame.from_dict(decisions, orient='index')


confidence_threshold = 0.7
xgboost_decisions_revised = revised_betting_strategy(best_model_xgb, test_data, confidence_threshold)
glm_decisions_revised = revised_betting_strategy(best_model_glm, test_data, confidence_threshold)
rf_decisions_revised = revised_betting_strategy(best_model_rf, test_data, confidence_threshold)

In [ ]:
# Function to evaluate strategy and calculate cumulative return based on odds
def evaluate_strategy_with_odds(decisions_df, true_results, match_df):
    total_bets = 0
    correct_bets = 0
    cumulative_return = 0

    for fixture_id, decision in decisions_df.iterrows():
        prediction = decision['prediction']

        # Skip if "no action"
        if prediction == "no action":
            continue

        true_result = true_results.get(fixture_id)
        
        # Filter for the specific fixture and minute to get the odds
        odds_row = match_df[
            (match_df['fixture_id'] == fixture_id) & (match_df['minute'] == decision['minute'])
        ].iloc[0]

        # Get the appropriate odds based on the prediction
        if prediction == "1":
            odd = odds_row['1']  # Home win odds
        elif prediction == "2":
            odd = odds_row['2']  # Away win odds
        else:
            odd = odds_row['X']  # Draw odds

        total_bets += 1

        if prediction == true_result:
            correct_bets += 1
            cumulative_return += odd  # Gain units based on odds if correct
        else:
            cumulative_return -= 1  # Lose 1 unit if incorrect

    accuracy = correct_bets / total_bets if total_bets > 0 else 0
    return {
        'Total Bets': total_bets,
        'Correct Bets': correct_bets,
        'Accuracy': accuracy,
        'Cumulative Return': cumulative_return
    }

# Example usage
y_true_results = test_data.groupby('fixture_id')['result'].first().to_dict()
xgboost_performance_with_odds = evaluate_strategy_with_odds(xgboost_decisions_revised, y_true_results, match_df)
glm_performance_with_odds = evaluate_strategy_with_odds(glm_decisions_revised, y_true_results, match_df)
rf_performance_with_odds = evaluate_strategy_with_odds(rf_decisions_revised, y_true_results, match_df)

# Display updated performance comparison
performance_with_odds_df = pd.DataFrame({
    'XGBoost': xgboost_performance_with_odds,
    'GLM': glm_performance_with_odds,
    'Random Forest': rf_performance_with_odds
}).T
performance_with_odds_df.columns = ['Total Bets', 'Correct Bets', 'Accuracy', 'Cumulative Return']
performance_with_odds_df.reset_index(inplace=True)
performance_with_odds_df.rename(columns={'index': 'Model'}, inplace=True)

print(performance_with_odds_df)

In [ ]:
# Function to print decisions for all models
import pandas as pd

def print_betting_decisions(decision_xgb, decision_glm, decision_rf):
    summary_df = pd.DataFrame({
        'XGBoost': decision_xgb['prediction'],
        'GLM': decision_glm['prediction'],
        'Random Forest': decision_rf['prediction'],
        'Minute (XGBoost)': decision_xgb['minute'],
        'Minute (GLM)': decision_glm['minute'],
        'Minute (RF)': decision_rf['minute']
    })
    # Filter rows where at least one model made a bet
    summary_df = summary_df[summary_df[['XGBoost', 'GLM', 'Random Forest']].ne("no action").any(axis=1)]
    
    print("\nSummary of Betting Decisions (First 20 Matches):")
    print(summary_df.head(20))  # Print the first 20 rows for a quick view

    return summary_df

# Example usage
betting_summary = print_betting_decisions(xgboost_decisions_revised, glm_decisions_revised, rf_decisions_revised)


Adam (2016): Used GLMs with L2 regularization to predict match scores. The study combined player-based features (e.g., previous match performances) and team-based features (e.g., FIFA ratings).Inspired the use of cumulative and momentum-based features like shots and goals, highlighting that both long-term and short-term trends are crucial.
Bunker and Thabtah (2019) reviewed sports outcome prediction methods and highlighted the importance of rolling statistics for live data. For soccer, features like “number of shots in the last 5 minutes” and “recent fouls” were emphasized to capture match dynamics.
Rolling window features like shots_last_5_home and fouls_last_5_home help track short-term performance.
Angelini et al. (2020) demonstrated that odds changes during a match reflect new market beliefs about the game's likely outcome. Their research underlined that sudden shifts in odds can indicate events like goals or red cards.
Odds-related features can act as “market signals” for feature engineering.
Eggels et al. (2016) Proposed adding binary features for key events (e.g., goals, red cards, substitutions). Their study proved that these events significantly impact in-game predictions.
Event indicators such as is_goal_home_recent and is_red_card_away_recent are engineered and inspired in thid domain.
Lastly Decroos et al. (2019) introduced “Soccer Action Models” to assess game states by combining cumulative actions and recent plays. Their work showed that momentum metrics (e.g., shots and dangerous attacks in a time window) significantly contribute to outcome predictions.
Momentum features engineered to provide a snapshot of recent dominance in the game.